# GLD vs. 10Y Real Yields: First Signal Hypothesis

**Author:** Jan  
**Last updated:** 2026-05-16

Tests the academic prior that GLD returns are negatively related to changes in the 10Y real yield (DFII10). If the relationship holds out-of-sample and is stable across sub-periods, this becomes the first component of the fundamental vertical's GLD allocation signal.

## Why monthly frequency?

The competition submits weekly, so a fair question is why this regression is at monthly frequency.

1. **The driver is intrinsically monthly.** The macro narrative we care about (real-yield trend changes the opportunity cost of holding gold) operates on the time scale of FRED's release calendar, not on day-to-day yield jitter. Most fundamental inputs (`CPILFESL`, `INDPRO`, `PAYEMS`, `UNRATE`) are published monthly to begin with.
2. **Higher frequency adds noise, not information.** `DFII10` is daily, but weekly changes are dominated by intraday yield volatility rather than meaningful macro shifts. A weekly regression would produce ~4x more observations with a slope biased toward zero.
3. **The signal updates monthly anyway.** Inside a calendar month no new macro data arrives, so a weekly-recomputed signal would be identical to the prior week's until the next FRED release. The weekly application in production (`src/fundamental.py` called every Friday in `notebooks/14`) just forward-fills the latest monthly signal to each Friday.

Bottom line: monthly is the frequency of the relationship; weekly is the cadence of the submission. The two don't have to match.

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import linregress

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import TRAIN_END
from src.data import load_etf_prices, load_fred_series

pd.options.display.float_format = "{:.4f}".format
np.random.seed(0)

## Data

GLD trades from 2004-11-18 onwards; DFII10 (10Y real yield from TIPS) is available from 2003-01-02. We align on month-end last values. Features:

- `DFII10_change`: monthly change in real yield, in percentage points
- `GLD_return`: monthly total return on GLD (auto-adjusted close)

**Filter to training period.** All regressions in this notebook are restricted to `<= TRAIN_END` (2024-12-31). The original validation used full-sample data; this is the disciplined re-fit on training only.

In [ ]:
real_yield = load_fred_series("DFII10", start="2004-11-18", end=str(TRAIN_END.date()))
gld_full = load_etf_prices("GLD", start="2004-11-18")["GLD"]
gld = gld_full[gld_full.index <= TRAIN_END]

daily = pd.DataFrame({"DFII10": real_yield, "GLD": gld})
monthly = daily.resample("ME").last().dropna()
monthly["DFII10_change"] = monthly["DFII10"].diff()
monthly["GLD_return"] = monthly["GLD"].pct_change()
clean = monthly.dropna()
clean.tail()

In [ ]:
clean[["DFII10_change", "GLD_return"]].describe()

## Analysis

In [ ]:
full = linregress(clean["DFII10_change"], clean["GLD_return"])

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(clean["DFII10_change"], clean["GLD_return"], alpha=0.5, s=24)

x_range = np.linspace(clean["DFII10_change"].min(), clean["DFII10_change"].max(), 100)
y_fit = full.intercept + full.slope * x_range
ax.plot(x_range, y_fit, color="crimson", linewidth=2,
        label=f"β = {full.slope:+.3f}, R² = {full.rvalue**2:.3f}, N = {len(clean)}")

ax.axhline(0, color="gray", alpha=0.3, linewidth=0.8)
ax.axvline(0, color="gray", alpha=0.3, linewidth=0.8)
ax.set_xlabel("Monthly change in 10Y real yield (DFII10), percentage points")
ax.set_ylabel("Monthly GLD return")
ax.set_title(f"GLD return vs. 10Y real yield change (training data, 2004-11 to {TRAIN_END.year}-{TRAIN_END.month:02d})")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print(f"Training-only regression (2004-11 to {TRAIN_END.date()}):")
print(f"  Slope (β):  {full.slope:+.4f}  (1%-pt real yield rise -> {full.slope*100:+.2f}% GLD return)")
print(f"  Intercept:  {full.intercept:+.4f}")
print(f"  R-squared:  {full.rvalue**2:.4f}")
print(f"  t-stat:     {full.slope / full.stderr:+.2f}")
print(f"  p-value:    {full.pvalue:.2e}")
print(f"  N obs:      {len(clean)}")

In [ ]:
periods = [
    ("2005-2011", "2005-01-01", "2011-12-31"),
    ("2012-2017", "2012-01-01", "2017-12-31"),
    ("2018-2024", "2018-01-01", "2024-12-31"),
]

rows = []
for label, start, end in periods:
    sub = clean.loc[start:end]
    if len(sub) < 12:
        continue
    res = linregress(sub["DFII10_change"], sub["GLD_return"])
    rows.append({
        "period": label,
        "n": len(sub),
        "slope": res.slope,
        "t_stat": res.slope / res.stderr,
        "R2": res.rvalue ** 2,
        "p_value": res.pvalue,
    })

pd.DataFrame(rows).set_index("period")

## Results

Populate after running. Key questions to answer from the cells above:

1. Is the slope negative and statistically significant (|t| > 2)?
2. Is the sign of the slope stable across all three sub-periods?
3. What is the R-squared? Single-input signals typically have low R-squared (1 to 15%) on monthly data, so the bar is "directionally right and stable", not "high explanatory power".

## Notes / next steps

The slope was negative and sign-stable across sub-periods, so the signal was carried forward:

- Signal construction (z-score → tanh tilt → baseline weight) in `notebooks/04_gld_signal_jan.ipynb`.
- Out-of-sample validation and KPIs in `notebooks/15_kpis_jan.ipynb`.
- Production code: `src/fundamental.py` (`get_weights` contract).

Complementary inputs (USD via DTWEXBGS, breakevens via T10YIE) and multi-driver extensions were tested and rejected; the production signal remains the single-driver DFII10 z-score tilt.